# Tutorial 09 — Efficient inference, compression, and deployment tradeoffs

[Lecture notes](../lectures/09_efficiency.md) · [Exercise sheet](../exercises/09_efficiency.md)

**40 minutes, CPU, no accounts or model downloads.** Requires Python 3.10+ and NumPy. These are mechanism experiments using authored or synthetic data, not frontier benchmark results. Restart the kernel and run all cells in order.

**Objective:** Quantize a matrix at two bit widths and estimate KV-cache growth with context length and head sharing.

Allocate 5 minutes to setup, 15 to the mechanism, 10 to interventions, and 10 to discussion. Record predictions before running. Complete the separate exercise sheet after class.


In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)


Python 3.13.1 NumPy 2.1.3 seed 42


## 1. Quantize and reconstruct

Per-output-channel scaling uses axis 0 for a din×dout matrix. This demonstrates approximation and theoretical storage, not packed low-bit kernels.

Before running: predict the output or the invariant being checked.


In [2]:
W=rng.normal(size=(64,32)); W[:,0]*=30
X=rng.normal(size=(128,64))
def quantize(W,bits,per_channel=True):
    qmax=2**(bits-1)-1
    maxabs=np.max(np.abs(W),axis=0,keepdims=True) if per_channel else np.max(np.abs(W))
    scale=np.where(maxabs==0,1,maxabs/qmax)
    q=np.clip(np.rint(W/scale),-qmax,qmax).astype(np.int8)
    return q,scale,q*scale
for bits in [4,8]:
    for pc in [False,True]:
        q,s,Wq=quantize(W,bits,pc)
        print(bits,"per_channel",pc,"weight MSE",np.mean((W-Wq)**2),"output MSE",np.mean((X@W-X@Wq)**2))
assert np.isfinite(quantize(np.zeros((2,3)),4)[2]).all()


4 per_channel False weight MSE 1.1978280597259148 output MSE 78.49535052190845
4 per_channel True weight MSE 0.23210387377363365 output MSE 15.42621428265472
8 per_channel False weight MSE 0.020517345746461427 output MSE 1.3266943933137862
8 per_channel True weight MSE 0.0006133384432108045 output MSE 0.0359336578001878


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 2. Account for KV cache

These estimates assume all layers cache the full sequence and fixed precision. Actual allocation may differ.

Before running: predict the output or the invariant being checked.


In [3]:
def kv_bytes(B,T,L,Hkv,dh,bytes_per_element=2): return 2*B*T*L*Hkv*dh*bytes_per_element
for T in [1024,4096,16384]:
    print("context",T,"8 KV heads MiB",kv_bytes(1,T,24,8,64)/2**20,"2 KV heads MiB",kv_bytes(1,T,24,2,64)/2**20)
assert kv_bytes(1,4096,24,8,64)==192*2**20
assert kv_bytes(2,8192,32,4,128)==2**30


context 1024 8 KV heads MiB 48.0 2 KV heads MiB 12.0
context 4096 8 KV heads MiB 192.0 2 KV heads MiB 48.0
context 16384 8 KV heads MiB 768.0 2 KV heads MiB 192.0


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 3. Storage estimates versus NumPy storage

The array uses int8 storage even for four-bit values; report this explicitly. Packing two values per byte would require another implementation.

Before running: predict the output or the invariant being checked.


In [4]:
q,s,_=quantize(W,4)
ideal_packed=W.size*4/8+s.size*4 # assume float32 scales
print("ideal packed bytes with float32 scales",ideal_packed)
print("actual NumPy bytes",q.nbytes+s.nbytes)
print("float32 baseline bytes",W.size*4)
assert q.nbytes==W.size


ideal packed bytes with float32 scales 1152.0
actual NumPy bytes 2304
float32 baseline bytes 8192


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## Practical discussion

1. Why does four-bit weight storage not make all inference memory four-bit?
2. Why can a throughput improvement worsen user latency?
3. What additional evidence is needed after measuring low weight MSE?


## Extension and submission

Time actual runs of `extensions/hf_text.py` on a declared device. Its wall time includes model loading unless you instrument generation separately; do not call it decode throughput.

See the [extension guide](../extensions/README.md) before running model downloads. Extensions require additional dependencies and are not necessary for full exercise credit.

Submit the completed notebook with a 150–250-word interpretation and the separate exercise answers. Include a baseline, one controlled change, data provenance, seed, and an honest limitation. Never replace measured results with expected trends.
